# Metodi numerici per ODE: esercitazioni svolte

Versione docente con implementazioni e risultati numerici. Compatibile con Google Colab; usa solo NumPy e Matplotlib.

Notazione: `y_n` è il valore esatto, `u_n` l'approssimazione numerica e `f_n=f(t_n,u_n)`.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt



plt.rcParams['figure.figsize'] = (8, 4.5)

plt.rcParams['axes.grid'] = True

## 1. Metodi a un passo

Problema: `y'=-y`, `y(0)=1`, soluzione esatta `y(t)=exp(-t)`. Si confrontano Euler, Heun e Runge–Kutta classico di ordine quattro per `h=0.2` su `[0, 2]`.

In [ ]:
def integrate_euler(rhs, initial_time, final_time, initial_value, step):

    step_count = int(round((final_time - initial_time) / step))

    time_grid = np.linspace(initial_time, final_time, step_count + 1)

    values = np.empty(step_count + 1)

    values[0] = initial_value

    for index in range(step_count):

        values[index + 1] = values[index] + step * rhs(time_grid[index], values[index])

    return time_grid, values





def integrate_heun(rhs, initial_time, final_time, initial_value, step):

    step_count = int(round((final_time - initial_time) / step))

    time_grid = np.linspace(initial_time, final_time, step_count + 1)

    values = np.empty(step_count + 1)

    values[0] = initial_value

    for index in range(step_count):

        time_n, value_n = time_grid[index], values[index]

        K1 = rhs(time_n, value_n)

        K2 = rhs(time_n + step, value_n + step * K1)

        values[index + 1] = value_n + 0.5 * step * (K1 + K2)

    return time_grid, values





def integrate_rk4(rhs, initial_time, final_time, initial_value, step):

    step_count = int(round((final_time - initial_time) / step))

    time_grid = np.linspace(initial_time, final_time, step_count + 1)

    values = np.empty(step_count + 1)

    values[0] = initial_value

    for index in range(step_count):

        time_n, value_n = time_grid[index], values[index]

        K1 = rhs(time_n, value_n)

        K2 = rhs(time_n + step / 2, value_n + step * K1 / 2)

        K3 = rhs(time_n + step / 2, value_n + step * K2 / 2)

        K4 = rhs(time_n + step, value_n + step * K3)

        values[index + 1] = value_n + step * (K1 + 2*K2 + 2*K3 + K4) / 6

    return time_grid, values



rhs_decay = lambda time, value: -value

method_functions = {'Euler': integrate_euler, 'Heun': integrate_heun, 'RK4': integrate_rk4}

time_grid = np.linspace(0.0, 2.0, 11)

exact_values = np.exp(-time_grid)

plt.plot(time_grid, exact_values, color='black', linewidth=2, label='Esatta')

for method_name, method_function in method_functions.items():

    method_time, method_values = method_function(rhs_decay, 0.0, 2.0, 1.0, 0.2)

    plt.plot(method_time, method_values, marker='o', label=method_name)

    print(f'{method_name}: errore massimo = {np.max(np.abs(method_values - exact_values)):.6e}')

plt.xlabel('t')

plt.ylabel('u(t)')

plt.legend()

plt.show()

## 2. Ordine di convergenza

Si usa l'errore massimo sui nodi e la pendenza della regressione di `log(errore)` contro `log(h)`. I valori osservati sono prossimi agli ordini teorici 1 per Euler, 2 per Heun e 4 per RK4.

In [ ]:
step_sizes = np.array([0.2, 0.1, 0.05, 0.025])

for method_name, method_function in method_functions.items():

    errors = []

    for step in step_sizes:

        method_time, method_values = method_function(rhs_decay, 0.0, 2.0, 1.0, step)

        errors.append(np.max(np.abs(method_values - np.exp(-method_time))))

    estimated_order = np.polyfit(np.log(step_sizes), np.log(errors), 1)[0]

    print(f'{method_name}: errori={np.array(errors)}, ordine stimato={estimated_order:.3f}')

    plt.loglog(step_sizes, errors, 'o-', label=method_name)

plt.xlabel('h')

plt.ylabel('Errore massimo')

plt.legend()

plt.show()

## 3. Stabilità assoluta di Euler

Per `y'=-10y`, il fattore è `R=1-10h`. Con `h=0.05`, `R=0.5`: decadimento monotono. Con `h=0.15`, `R=-0.5`: decadimento oscillante. Con `h=0.25`, `R=-1.5`: instabilità numerica.

In [ ]:
decay_rate = 10.0

rhs_stiff = lambda time, value: -decay_rate * value

for step in [0.05, 0.15, 0.25]:

    amplification = 1.0 - decay_rate * step

    time_grid, values = integrate_euler(rhs_stiff, 0.0, 1.0, 1.0, step)

    print(f'h={step:.2f}, R={amplification:.2f}, stabile={abs(amplification) < 1.0}')

    plt.plot(time_grid, values, marker='o', label=f'Euler h={step}')

time_exact = np.linspace(0.0, 1.0, 300)

plt.plot(time_exact, np.exp(-decay_rate * time_exact), color='black', label='Esatta')

plt.xlabel('t')

plt.ylabel('u(t)')

plt.legend()

plt.show()

## 4. Adams–Bashforth 2 e BDF2

AB2 è esplicito. BDF2 è implicito nel caso generale; per `y'=-a y` la sua equazione si risolve direttamente. Heun fornisce il dato iniziale aggiuntivo `u_1`.

In [ ]:
def integrate_adams_bashforth_2(rhs, initial_time, final_time, initial_value, step):

    step_count = int(round((final_time - initial_time) / step))

    time_grid = np.linspace(initial_time, final_time, step_count + 1)

    values = np.empty(step_count + 1)

    values[0] = initial_value

    first_slope = rhs(time_grid[0], values[0])

    predictor = values[0] + step * first_slope

    second_slope = rhs(time_grid[1], predictor)

    values[1] = values[0] + 0.5 * step * (first_slope + second_slope)

    for index in range(1, step_count):

        f_n = rhs(time_grid[index], values[index])

        f_previous = rhs(time_grid[index - 1], values[index - 1])

        values[index + 1] = values[index] + 0.5 * step * (3*f_n - f_previous)

    return time_grid, values





def integrate_bdf2_decay(initial_time, final_time, initial_value, step, decay_rate):

    step_count = int(round((final_time - initial_time) / step))

    time_grid = np.linspace(initial_time, final_time, step_count + 1)

    values = np.empty(step_count + 1)

    values[0] = initial_value

    first_slope = -decay_rate * values[0]

    predictor = values[0] + step * first_slope

    values[1] = values[0] + 0.5 * step * (first_slope - decay_rate * predictor)

    for index in range(1, step_count):

        values[index + 1] = (4*values[index] - values[index - 1]) / (3 + 2*step*decay_rate)

    return time_grid, values



step = 0.1

time_grid, ab_values = integrate_adams_bashforth_2(rhs_decay, 0.0, 2.0, 1.0, step)

time_grid, bdf_values = integrate_bdf2_decay(0.0, 2.0, 1.0, step, 1.0)

time_grid, rk_values = integrate_rk4(rhs_decay, 0.0, 2.0, 1.0, step)

exact_values = np.exp(-time_grid)

for method_name, values in [('AB2', ab_values), ('BDF2', bdf_values), ('RK4', rk_values)]:

    print(f'{method_name}: errore massimo = {np.max(np.abs(values - exact_values)):.6e}')

    plt.plot(time_grid, values, marker='o', label=method_name)

plt.plot(time_grid, exact_values, color='black', label='Esatta')

plt.xlabel('t')

plt.ylabel('u(t)')

plt.legend()

plt.show()